In [2]:
import pandas as pd
import numpy as np
import re
import unicodedata
import os
import duckdb

#### Leitura & transformação

In [3]:
# Definindo a estrutura de caminhos relativos do projeto
BRONZE_DIR = "../data/bronze"
SILVER_DIR = "../data/silver"

In [4]:
def ler_nomes_arquivos_bronze():
    """
    Lê os nomes dos arquivos na pasta bronze e retorna uma lista de nomes de arquivos.
    """
    return [f for f in os.listdir(BRONZE_DIR) if os.path.isfile(os.path.join(BRONZE_DIR, f))]

In [5]:
lista_arquivos = ler_nomes_arquivos_bronze()

In [6]:
def transformar_csv_para_parquet(lista_arquivos):
    """
    Função para transformar um arquivo CSV em Parquet.
    
    Parâmetros:
    lista_arquivos (list): Lista de nomes dos arquivos CSV de entrada.
    """

    for arquivo in lista_arquivos:
        # Lendo o arquivo CSV
        df = pd.read_csv(os.path.join(BRONZE_DIR, arquivo))
        
        # Definindo o nome do arquivo Parquet de saída
        nome_arquivo_parquet = os.path.splitext(arquivo)[0] + '.parquet'
        
        # Salvando o DataFrame como Parquet
        df.to_parquet(os.path.join(SILVER_DIR, nome_arquivo_parquet), index=False)
        
        print(f"Arquivo {arquivo} transformado e salvo em data/silver")


In [7]:
transformar_csv_para_parquet(lista_arquivos)

Arquivo br_bd_diretorios_brasil_cnae_2.csv transformado e salvo em data/silver
Arquivo br_bd_diretorios_brasil_municipio.csv transformado e salvo em data/silver
Arquivo br_bndes_operacoes_contratadas_operacoes_nao_automaticas.csv transformado e salvo em data/silver


In [8]:
def ler_nomes_arquivos_silver():
    """
    Lê os nomes dos arquivos na pasta silver e retorna uma lista de nomes de arquivos.
    """
    return [f for f in os.listdir(SILVER_DIR) if os.path.isfile(os.path.join(SILVER_DIR, f))]

ler_nomes_arquivos_silver()

['bnds_silver.parquet',
 'br_bd_diretorios_brasil_cnae_2.parquet',
 'br_bd_diretorios_brasil_municipio.parquet',
 'br_bndes_operacoes_contratadas_operacoes_nao_automaticas.parquet']

In [9]:
operacoes = pd.read_parquet(os.path.join(SILVER_DIR, 'br_bndes_operacoes_contratadas_operacoes_nao_automaticas.parquet'))

operacoes.info()

<class 'pandas.DataFrame'>
RangeIndex: 23483 entries, 0 to 23482
Data columns (total 39 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   razao_social_cliente                     23483 non-null  str    
 1   cnpj_cliente                             23474 non-null  float64
 2   descricao_projeto                        23483 non-null  str    
 3   sigla_uf                                 23483 non-null  str    
 4   nome_municipio                           23483 non-null  str    
 5   id_municipio                             15762 non-null  float64
 6   id_contrato                              23483 non-null  int64  
 7   data_contratacao                         23483 non-null  str    
 8   valor_contratado                         23483 non-null  float64
 9   valor_desembolsado                       23483 non-null  float64
 10  tipo_fonte_recursos                      22647 non-null  

#### Tratamento de valores nulos

In [10]:
def verificar_valores_nulos(df):
    """Verifica se um DataFrame contém valores nulos.

    Parâmetros:
        df (DataFrame): O DataFrame a ser verificado.

    Retorna:
        False | DataFrame: False se não houver nulos; DataFrame com análise se houver.
    """
    total_nulos = df.isnull().sum().sort_values(ascending=False)
    total_nulos = total_nulos[total_nulos > 0]

    if total_nulos.empty:
        return False

    total_nulos_percent = ((total_nulos / df.shape[0]) * 100).round(2)
    return pd.DataFrame({'Total Nulls': total_nulos, '%': total_nulos_percent})

In [11]:
verificar_valores_nulos(operacoes)

,Total Nulls,%
tipo_excepcionalidade,23265,99.07
cnpj_instituicao_financeira_credenciada,19412,82.66
nome_instituicao_financeira_credenciada,19412,82.66
id_municipio,7721,32.88
subclasse_cnae,5507,23.45
tipo_fonte_recursos,836,3.56
classe_cnae,497,2.12
situacao_contrato,268,1.14
grupo_cnae,255,1.09
cnpj_cliente,9,0.04


In [12]:
#categorias existentes na coluna tipo_excepcionalidade
operacoes["tipo_excepcionalidade"].value_counts(dropna=False)

tipo_excepcionalidade
NaN                                                                                  23265
CONDIÇÕES DE CRÉDITO                                                                    81
CONDIÇÕES FINANCEIRAS E OPERACIONAIS                                                    68
COBRANÇA DE COMISSÕES                                                                   47
CONDIÇÕES FINANCEIRAS E OPERACIONAIS /CONDIÇÕES DE CRÉDITO                              17
GARANTIAS                                                                                3
CONDIÇÕES FINANCEIRAS E OPERACIONAIS /COBRANÇA DE COMISSÕES /CONDIÇÕES DE CRÉDITO        2
Name: count, dtype: int64

**Regra de negócio:** Se _tipo_excepcionalidade_
 está nulo, significa que a operação seguiu o fluxo padrão (sem exceção). Então, podemos realizar a classificação 0 = não, e 1 = sim.

In [13]:
#Criando uma nova coluna para indicar se a operação possui ou não excepcionalidade
operacoes["tem_excepcionalidade"] = operacoes["tipo_excepcionalidade"].notna().astype(int)

In [14]:
operacoes = operacoes.drop(columns=["tipo_excepcionalidade"])

In [15]:
operacoes.isnull().sum().sort_values(ascending=False).head(10)

nome_instituicao_financeira_credenciada    19412
cnpj_instituicao_financeira_credenciada    19412
id_municipio                                7721
subclasse_cnae                              5507
tipo_fonte_recursos                          836
classe_cnae                                  497
situacao_contrato                            268
grupo_cnae                                   255
cnpj_cliente                                   9
id_contrato                                    0
dtype: int64

**Regra de Négocio:** Se o CNPJ da instituição financeira credenciada estiver nulo, podemos entender que a operação foi realizada diretamente com o BNDES, sem intermediação de uma instituição financeira. Portanto, vamos preencher esses valores nulos com a string "OPERAÇÃO DIRETA".

In [16]:
operacoes["cnpj_instituicao_financeira_credenciada"] = operacoes["cnpj_instituicao_financeira_credenciada"].fillna("0000000000000.0")
operacoes["nome_instituicao_financeira_credenciada"] = operacoes["nome_instituicao_financeira_credenciada"].fillna("OPERAÇÃO DIRETA")
print(f"{operacoes[['cnpj_instituicao_financeira_credenciada']].dtypes}")

cnpj_instituicao_financeira_credenciada    object
dtype: object


In [17]:
#Verficando colunas restantes para tratamento de valores nulos
operacoes.isnull().sum().sort_values(ascending=False).head(10)

id_municipio           7721
subclasse_cnae         5507
tipo_fonte_recursos     836
classe_cnae             497
situacao_contrato       268
grupo_cnae              255
cnpj_cliente              9
id_contrato               0
nome_municipio            0
sigla_uf                  0
dtype: int64

In [18]:
print(f"{operacoes[['cnpj_cliente']].dtypes}")

cnpj_cliente    float64
dtype: object


In [19]:
#preenchendo os valores nulos
operacoes["id_municipio"] = operacoes["id_municipio"].fillna("NÃO INFORMADO")
operacoes["cnpj_cliente"] = operacoes["cnpj_cliente"].fillna("000000000000.0")
operacoes["situacao_contrato"] = operacoes["situacao_contrato"].fillna("OUTROS")
operacoes["tipo_fonte_recursos"] = operacoes["tipo_fonte_recursos"].fillna("OUTROS")

In [20]:
#removendo colunas cnae desnecessárias
operacoes = operacoes.drop(columns=["classe_cnae","subclasse_cnae","grupo_cnae","divisao_cnae","secao_cnae"])

In [21]:
verificar_valores_nulos(operacoes)

False

#### Validação dos tipos de dados e Enriquecimento

In [22]:
#Verificando os tipos de dados das colunas
operacoes.dtypes

razao_social_cliente                           str
cnpj_cliente                                object
descricao_projeto                              str
sigla_uf                                       str
nome_municipio                                 str
id_municipio                                object
id_contrato                                  int64
data_contratacao                               str
valor_contratado                           float64
valor_desembolsado                         float64
tipo_fonte_recursos                            str
custo_financeiro                               str
taxa_juros                                 float64
prazo_carencia                               int64
prazo_amortizacao                            int64
modalidade_apoio                               str
forma_apoio                                    str
produto                                        str
tipo_instrumento_financeiro                    str
indicador_inovacao             

In [23]:
#consultando colunas de data
cols_data = [c for c in operacoes.columns if "data" in c]
cols_data
operacoes[cols_data].dtypes

data_contratacao    str
data_apuracao       str
dtype: object

In [24]:
#Ajuste dos tipos de dados das colunas de data
operacoes[cols_data] = operacoes[cols_data].apply(pd.to_datetime, errors='coerce')
operacoes[cols_data].dtypes

data_contratacao    datetime64[us]
data_apuracao       datetime64[us]
dtype: object

### Limpeza e padronização de textos

In [25]:
def normalize_text(value):
    if pd.isna(value):
        return value
    text = str(value).strip().upper()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"\s+", " ", text)
    return text

In [26]:
def padronizar_texto(df, cols=None):
    df_limpo = df.copy()
    if cols is None:
        cols = df_limpo.select_dtypes(include=["object", "string"]).columns
    for col in cols:
        df_limpo[col] = df_limpo[col].map(normalize_text)
    return df_limpo

In [27]:
operacoes = padronizar_texto(operacoes)

### Feature Engineering

In [28]:
print(operacoes['porte_cliente'].unique())
print(operacoes['modalidade_apoio'].unique())
print(operacoes['forma_apoio'].unique())
print(operacoes['natureza_cliente'].unique())

<ArrowStringArray>
['GRANDE', 'SEM PORTE', 'MEDIA', 'PEQUENA', 'MICRO']
Length: 5, dtype: str
<ArrowStringArray>
['REEMBOLSAVEL', 'NAO REEMBOLSAVEL']
Length: 2, dtype: str
<ArrowStringArray>
['INDIRETA', 'DIRETA']
Length: 2, dtype: str
<ArrowStringArray>
[                                         'PRIVADA',
 'ADMINISTRACAO PUBLICA DIRETA - GOVERNO MUNICIPAL',
  'ADMINISTRACAO PUBLICA DIRETA - GOVERNO ESTADUAL',
                                 'PUBLICA INDIRETA',
   'ADMINISTRACAO PUBLICA DIRETA - GOVERNO FEDERAL']
Length: 5, dtype: str


In [29]:
def aplicar_feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """Aplica as transformações da camada Silver no DataFrame de operações do BNDES."""
    
    # Cria uma cópia para preservar a base original
    df_out = df.copy()

    # 1. Extração do ano e mês da data de contratação
    df_out["ano_contratacao"] = df_out["data_contratacao"].dt.year
    df_out["mes_contratacao"] = df_out["data_contratacao"].dt.month

    # 2. Calculo do prazo total em meses
    df_out["prazo_total_meses"] = df_out["prazo_carencia"].fillna(0) + df_out[
        "prazo_amortizacao"
    ].fillna(0)

    # 3. Categorização da taxa de juros
    condicoes_taxa = [
        df_out["taxa_juros"] == 0,
        df_out["taxa_juros"] < 5,
        (df_out["taxa_juros"] >= 5) & (df_out["taxa_juros"] <= 10),
        df_out["taxa_juros"] > 10,
    ]
    categorias_taxa = ["SEM TAXA", "BAIXA", "MEDIA", "ALTA"]

    df_out["grupo_taxa_juros"] = np.select(
        condicoes_taxa, categorias_taxa, default="SEM TAXA"
    )

    # 6. Codificação ordinal da taxa de juros
    mapa_taxa ={
        "SEM TAXA": 0,
        "BAIXA": 1,
        "MEDIA": 2,
        "ALTA": 3
    }
    df_out["grupo_taxa_juros_ordinal"] = df_out["grupo_taxa_juros"].map(mapa_taxa)

    # 4. Flags binárias
    df_out["flag_apoio_direto"] = (df_out["forma_apoio"] == "DIRETA").astype(int)
    df_out["flag_reembolsavel"] = (df_out["modalidade_apoio"] == "REEMBOLSAVEL").astype(int)
    df_out["flag_cliente_publico"] = (df_out["natureza_cliente"].fillna("").str.contains("PUBLICA|PUBLICO", regex=True).astype(int))

    # 5. Quantidade de palavras da descrição
    df_out["qtd_palavras_descricao"] = (
        df_out["descricao_projeto"].fillna("").str.split().str.len()
    )

    # 6. Codificação ordinal do porte do cliente
    mapa_porte = {
        "SEM PORTE": 0,
        "MICRO": 1,
        "PEQUENA": 2,
        "MEDIA": 3,
        "GRANDE": 4,
    }
    df_out["porte_cliente_ordinal"] = (df_out["porte_cliente"].map(mapa_porte))

    return df_out

In [30]:
operacoes = aplicar_feature_engineering(operacoes)
display(operacoes.head())

,razao_social_cliente,cnpj_cliente,descricao_projeto,sigla_uf,nome_municipio,id_municipio,id_contrato,data_contratacao,valor_contratado,valor_desembolsado,...,ano_contratacao,mes_contratacao,prazo_total_meses,grupo_taxa_juros,grupo_taxa_juros_ordinal,flag_apoio_direto,flag_reembolsavel,flag_cliente_publico,qtd_palavras_descricao,porte_cliente_ordinal
0,COPACOL-COOPERATIVA AGROINDUSTRIAL CONSOLATA,76093731000190.0,INVESTIMENTOS EM AMPLIACAO DA CAPACIDADE DE RE...,PR,SEM MUNICIPIO,NAO INFORMADO,9203361,2009-06-30,35900000.0,35900000.0,...,2009,6,108,MEDIA,2,0,1,0,44,4
1,LAR COOPERATIVA AGROINDUSTRIAL,77752293000198.0,DIRETA - AMPLICACAO DE LINHA DE ABATE DE FRANG...,PR,MATELANDIA,4115606.0,9203571,2009-06-29,66998000.0,66998000.0,...,2009,6,108,MEDIA,2,1,1,0,24,4
2,LAR COOPERATIVA AGROINDUSTRIAL,77752293000198.0,DIRETA - AMPLICACAO DE LINHA DE ABATE DE FRANG...,MS,SEM MUNICIPIO,NAO INFORMADO,9203581,2009-06-30,20897000.0,20897000.0,...,2009,6,108,MEDIA,2,0,1,0,24,4
3,COCARI - COOPERATIVA AGROPECUARIA E INDUSTRIAL,78956968000183.0,SUPLEMENTACAO DE RECURSOS A IMPLANTACAO DE UM ...,PR,MANDAGUARI,4114203.0,9204441,2009-06-30,37000000.0,37000000.0,...,2009,6,120,MEDIA,2,0,1,0,45,4
4,COCARI - COOPERATIVA AGROPECUARIA E INDUSTRIAL,78956968000183.0,SUPLEMENTACAO DE RECURSOS A IMPLANTACAO DE UM ...,PR,MANDAGUARI,4114203.0,9204451,2009-10-09,50000000.0,50000000.0,...,2009,10,120,MEDIA,2,0,1,0,45,4


In [ ]:
# Verificando os valores únicos das colunas categóricas e suas flags correspondentes
print(operacoes.groupby("forma_apoio")["flag_apoio_direto"].first())
print(operacoes.groupby("modalidade_apoio")["flag_reembolsavel"].first())
print(operacoes.groupby("natureza_cliente")["flag_cliente_publico"].first())
print(operacoes.groupby("porte_cliente")["porte_cliente_ordinal"].first())
print(operacoes.groupby("grupo_taxa_juros")["grupo_taxa_juros_ordinal"].first())

forma_apoio
DIRETA      1
INDIRETA    0
Name: flag_apoio_direto, dtype: int64
modalidade_apoio
NAO REEMBOLSAVEL    0
REEMBOLSAVEL        1
Name: flag_reembolsavel, dtype: int64
natureza_cliente
ADMINISTRACAO PUBLICA DIRETA - GOVERNO ESTADUAL     1
ADMINISTRACAO PUBLICA DIRETA - GOVERNO FEDERAL      1
ADMINISTRACAO PUBLICA DIRETA - GOVERNO MUNICIPAL    1
PRIVADA                                             0
PUBLICA INDIRETA                                    1
Name: flag_cliente_publico, dtype: int64
porte_cliente
GRANDE       4
MEDIA        3
MICRO        1
PEQUENA      2
SEM PORTE    0
Name: porte_cliente_ordinal, dtype: int64
grupo_taxa_juros
ALTA        3
BAIXA       1
MEDIA       2
SEM TAXA    0
Name: grupo_taxa_juros_ordinal, dtype: int64
tem_excepcionalidade
0    0
1    1
Name: tem_excepcionalidade, dtype: int64


### Validação de valores totais: (Silver x Raw) `Valor_Contrado` & `Valor_desembolsado`

In [32]:
silver_operacoes = operacoes.copy()

total = silver_operacoes[["valor_contratado", "valor_desembolsado"]].sum().round(2)
total_silver= total.apply(lambda x: f"{x:,.2f}")
total_silver

valor_contratado      1,229,184,483,691.93
valor_desembolsado      952,046,450,761.74
dtype: str

In [33]:
# Ler os dados brutos
raw = pd.read_csv(
    os.path.join(BRONZE_DIR, 'br_bndes_operacoes_contratadas_operacoes_nao_automaticas.csv'),
    dtype=str
)

cols = ["valor_contratado", "valor_desembolsado"]

In [34]:
total = raw[cols].apply(pd.to_numeric, errors="coerce").sum().round(2)

total_raw = total.apply(lambda x: f"{x:,.2f}")
total_raw

valor_contratado      1,229,184,483,691.93
valor_desembolsado      952,046,450,761.74
dtype: str